In [37]:
using LowLevelFEM

In [38]:
structured_rect_mesh(lx=2, n=1)
openPreProcessor()

mat = Material("surface", E=2e5, ν=0.3)
Pu = Problem([mat], type=:VectorField, dim=2, field=:u, rhs_field=:f);

In [39]:
E = mat.E
ν = mat.ν

0.3

In [40]:
D = E / (1 - ν^2) * [1 ν 0; ν 1 0; 0 0 (1-ν)/2]

3×3 Matrix{Float64}:
     2.1978e5  65934.1           0.0
 65934.1           2.1978e5      0.0
     0.0           0.0       76923.1

In [41]:
D = E / (1 + ν) / (1 - 2ν) * [1-ν ν 0; ν 1-ν 0; 0 0 (1-2ν)/2]

3×3 Matrix{Float64}:
 2.69231e5  1.15385e5      0.0
 1.15385e5  2.69231e5      0.0
 0.0        0.0        76923.1

In [42]:
K = ∫(SymGrad(Pu) ⋅ D ⋅ SymGrad(Pu), Ω="surface")
K[:, :]

12×12 SparseArrays.SparseMatrixCSC{Float64, Int64} with 107 stored entries:
      1.15385e5   48076.9        …  -57692.3          -48076.9
  48076.9             1.15385e5     -48076.9          -57692.3
       ⋅               ⋅            -57692.3           48076.9
       ⋅               ⋅             48076.9          -57692.3
       ⋅               ⋅            -76923.1            9615.38
       ⋅               ⋅         …   -9615.38          19230.8
  19230.8          9615.38          -76923.1           -9615.38
  -9615.38       -76923.1             9615.38          19230.8
 -76923.1         -9615.38           38461.5                ⋅ 
   9615.38        19230.8                 ⋅               -1.53846e5
 -57692.3        -48076.9        …       2.30769e5          ⋅ 
 -48076.9        -57692.3               -7.27596e-12       2.30769e5

In [43]:
fy = [0, -1]

2-element Vector{Int64}:
  0
 -1

In [44]:
f = ∫(Pu ⋅ fy, Γ="right")
DoFs(f)

12×1 Matrix{Float64}:
  0.0
  0.0
  0.0
 -0.5
  0.0
 -0.5
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0

In [45]:
supp = BoundaryCondition("left", ux=0, uy=0)

BoundaryCondition("left", nothing, Dict{Symbol, Union{Function, Number, ScalarField}}(:uy => 0, :ux => 0))

In [46]:
u = solveField(K, f, support=[supp])
DoFs(u)

12×1 Matrix{Float64}:
  0.0
  0.0
 -3.466666666666661e-5
 -0.00011266666666666643
  3.466666666666661e-5
 -0.00011266666666666646
  0.0
  0.0
 -2.5999999999999954e-5
 -3.89999999999999e-5
  2.5999999999999948e-5
 -3.89999999999999e-5

In [ ]:
showDoFResults(u, name="u", visible=true)
showDoFResults(u, name="u", factor=1000)

1

In [48]:
εx = ∂x(u[1])
εy = ∂y(u[2])
γxy = ∂y(u[1]) + ∂x(u[2])

ε = [εx, εy, γxy];

In [49]:
σ = D * ε;

In [50]:
showElementResults(σ[1], name="σx")
showElementResults(σ[1], name="σx", smooth=true)
showElementResults(σ[2], name="σy")
showElementResults(σ[3], name="γxy")

5

In [51]:
openPostProcessor()